# Merge independent metadynamics replica PMFs

This notebook:

- reads 2 or more **replica PMF** files
- checks that all PMFs share the same 2D CV grid
- merges them by **Boltzmann/probability averaging**
- writes:
  - merged PMF

Use this for **independent replicas**, not for chunks/restarts of one continuous run.


In [1]:
from pathlib import Path
import fnmatch
import numpy as np

# ============================================================
# User settings
# ============================================================
system_names = ["apo_l", "apo_p", "holo_l", "holo_p"]

temperature = 310.0  # K
method = "boltzmann"  # "boltzmann" or "arithmetic"

search_dir = Path(".")  # folder containing the PMF files

# ============================================================
# Constants
# ============================================================
kB = 0.0019872041  # kcal/mol/K
beta = 1.0 / (kB * temperature)

# ============================================================
# Helpers
# ============================================================
def parse_header(path):
    header_lines = []
    with open(path, "r") as fh:
        for line in fh:
            if line.startswith("#"):
                header_lines.append(line.rstrip("\n"))
            elif line.strip() == "":
                continue
            else:
                break

    info = {
        "raw_header": header_lines,
        "dims": None,
        "x_min": None, "x_step": None, "nx": None, "x_periodic": None,
        "y_min": None, "y_step": None, "ny": None, "y_periodic": None,
    }

    if len(header_lines) >= 3:
        try:
            info["dims"] = int(header_lines[0].replace("#", "").strip())

            x_tokens = header_lines[1].replace("#", "").split()
            y_tokens = header_lines[2].replace("#", "").split()

            info["x_min"] = float(x_tokens[0])
            info["x_step"] = float(x_tokens[1])
            info["nx"] = int(float(x_tokens[2]))
            info["x_periodic"] = int(float(x_tokens[3]))

            info["y_min"] = float(y_tokens[0])
            info["y_step"] = float(y_tokens[1])
            info["ny"] = int(float(y_tokens[2]))
            info["y_periodic"] = int(float(y_tokens[3]))
        except Exception:
            pass

    return info


def read_pmf(path):
    path = Path(path)
    arr = np.loadtxt(path, comments="#")

    if arr.ndim != 2 or arr.shape[1] < 3:
        raise ValueError(f"{path} does not look like a 3-column PMF file.")

    x = arr[:, 0]
    y = arr[:, 1]
    f = arr[:, 2]

    x_unique = np.unique(x)
    y_unique = np.unique(y)

    nx = len(x_unique)
    ny = len(y_unique)

    if nx * ny != len(arr):
        raise ValueError(
            f"{path}: grid is incomplete or duplicated. "
            f"Expected {nx * ny} rows from unique x/y values, got {len(arr)}."
        )

    ix = {val: i for i, val in enumerate(x_unique)}
    iy = {val: j for j, val in enumerate(y_unique)}

    F = np.full((nx, ny), np.nan)
    for xv, yv, fv in arr[:, :3]:
        F[ix[xv], iy[yv]] = fv

    if np.isnan(F).any():
        raise ValueError(f"{path}: missing values on the PMF grid.")

    header = parse_header(path)

    return {
        "path": str(path),
        "x": x_unique,
        "y": y_unique,
        "F": F,
        "header": header,
    }


def same_grid(a, b, atol=1e-8):
    return (
        len(a["x"]) == len(b["x"])
        and len(a["y"]) == len(b["y"])
        and np.allclose(a["x"], b["x"], atol=atol, rtol=0.0)
        and np.allclose(a["y"], b["y"], atol=atol, rtol=0.0)
    )


def write_pmf(path, xvals, yvals, F, header_template=None):
    xvals = np.asarray(xvals)
    yvals = np.asarray(yvals)
    F = np.asarray(F)

    dx = float(np.median(np.diff(xvals))) if len(xvals) > 1 else 1.0
    dy = float(np.median(np.diff(yvals))) if len(yvals) > 1 else 1.0

    x_min = xvals[0] - dx / 2.0
    y_min = yvals[0] - dy / 2.0
    nx = len(xvals)
    ny = len(yvals)

    x_periodic = 1
    y_periodic = 0

    if header_template is not None:
        x_periodic = header_template.get("x_periodic", x_periodic)
        y_periodic = header_template.get("y_periodic", y_periodic)
        if header_template.get("x_step") is not None:
            dx = header_template["x_step"]
        if header_template.get("y_step") is not None:
            dy = header_template["y_step"]
        if header_template.get("x_min") is not None:
            x_min = header_template["x_min"]
        if header_template.get("y_min") is not None:
            y_min = header_template["y_min"]

    with open(path, "w") as fh:
        fh.write("# 2\n")
        fh.write(f"# {x_min: .14e} {dx: .14e} {nx:9d} {x_periodic:d}\n")
        fh.write(f"# {y_min: .14e} {dy: .14e} {ny:9d} {y_periodic:d}\n\n")

        for i, xv in enumerate(xvals):
            for j, yv in enumerate(yvals):
                fh.write(f"{xv: .14e} {yv: .14e} {F[i, j]: .14e}\n")


def shift_min_to_zero(F):
    return F - np.nanmin(F)


def merge_pmfs(F_stack, beta, method="boltzmann"):
    if method == "boltzmann":
        P = np.exp(-beta * F_stack)
        Pavg = np.mean(P, axis=0)
        Fm = -(1.0 / beta) * np.log(Pavg)
    elif method == "arithmetic":
        Fm = np.mean(F_stack, axis=0)
    else:
        raise ValueError("method must be 'boltzmann' or 'arithmetic'")
    return shift_min_to_zero(Fm)


def find_input_pmfs_for_system(system_name, directory):
    excluded_prefixes = (
        "merged_replicas_boltzmann_",
        "replica_stddev_map_",
        "replica_range_map_",
    )

    matches = []
    for path in sorted(directory.glob("*.pmf")):
        name = path.name

        if not fnmatch.fnmatch(name, f"{system_name}*.pmf"):
            continue

        if name.startswith(excluded_prefixes):
            continue

        matches.append(path)

    return matches


def print_header(title):
    print("=" * 100)
    print(title)
    print("=" * 100)


def process_system(system_name, pmf_paths):
    merged_output = f"merged_replicas_boltzmann_{system_name}.pmf"

    pmfs = [read_pmf(p) for p in pmf_paths]

    ref = pmfs[0]
    for p in pmfs[1:]:
        if not same_grid(ref, p):
            raise ValueError(
                f"Grid mismatch in system {system_name}:\n"
                f"  {ref['path']}\n"
                f"  {p['path']}\n"
                "Interpolate to a common grid before merging."
            )

    xvals = ref["x"]
    yvals = ref["y"]
    header_template = ref["header"]

    F_shifted = np.stack([shift_min_to_zero(p["F"]) for p in pmfs], axis=0)

    F_merged = merge_pmfs(F_shifted, beta=beta, method=method)

    write_pmf(merged_output, xvals, yvals, F_merged, header_template=header_template)

    print_header(system_name)

    print("Input replica PMFs:")
    for p in pmf_paths:
        print(f"   {p.name}")

    print("\nWrote:")
    print(f"   {merged_output}")
    print("\n")


# ============================================================
# Run all systems
# ============================================================
for system_name in system_names:
    matched_pmfs = find_input_pmfs_for_system(system_name, search_dir)

    try:
        if len(matched_pmfs) < 2:
            raise ValueError(
                f"Found fewer than 2 input PMF files for {system_name}: "
                f"{[p.name for p in matched_pmfs]}"
            )

        process_system(system_name, matched_pmfs)

    except Exception as e:
        print_header(system_name)
        print(f"ERROR: {e}\n")

apo_l
Input replica PMFs:
   apo_l_meta_rep1.pmf
   apo_l_meta_rep2.pmf

Wrote:
   merged_replicas_boltzmann_apo_l.pmf


apo_p
Input replica PMFs:
   apo_p_meta_rep1.pmf
   apo_p_meta_rep2.pmf

Wrote:
   merged_replicas_boltzmann_apo_p.pmf


holo_l
Input replica PMFs:
   holo_l_meta_rep1.pmf
   holo_l_meta_rep2.pmf

Wrote:
   merged_replicas_boltzmann_holo_l.pmf


holo_p
Input replica PMFs:
   holo_p_meta_rep1.pmf
   holo_p_meta_rep2.pmf

Wrote:
   merged_replicas_boltzmann_holo_p.pmf


